# Credit Card Fraud Detection
**Ensemble pipeline: Isolation Forest + XGBoost with SMOTE**

This notebook walks through the full pipeline:
1. Exploratory Data Analysis (EDA)
2. Feature Engineering
3. Anomaly scoring with Isolation Forest
4. Oversampling with SMOTE
5. XGBoost training with threshold optimisation
6. Evaluation & visualisation

**Dataset:** [Kaggle Credit Card Fraud Detection](https://www.kaggle.com/datasets/mlg-ulb/creditcardfraud)  
Download `creditcard.csv` and place it in the `data/` folder.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.ensemble import IsolationForest
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    classification_report, confusion_matrix, roc_auc_score,
    average_precision_score, precision_recall_curve, roc_curve
)
from imblearn.over_sampling import SMOTE
from xgboost import XGBClassifier

sns.set_theme(style='whitegrid')
plt.rcParams['figure.dpi'] = 120

import sys; sys.path.insert(0, '../src')
from fraud_detection import engineer_features

## 1. Load Data

In [ ]:
df = pd.read_csv('../data/creditcard.csv')
print(f'Shape: {df.shape}')
print(f'Fraud rate: {df["Class"].mean()*100:.4f}%')
df.head()

## 2. EDA

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Class imbalance
counts = df['Class'].value_counts()
axes[0].bar(['Legitimate', 'Fraud'], counts, color=['#2563EB', '#DC2626'])
axes[0].set_title('Class Distribution (severe imbalance)')
axes[0].set_ylabel('Count')
for i, v in enumerate(counts):
    axes[0].text(i, v + 100, f'{v:,}', ha='center', fontweight='bold')

# Amount boxplot
df.boxplot(column='Amount', by='Class', ax=axes[1])
axes[1].set_title('Transaction Amount by Class')
axes[1].set_xticklabels(['Legitimate', 'Fraud'])
plt.suptitle('')
plt.tight_layout()
plt.show()

In [ ]:
# Correlation heatmap (fraud subset)
fraud_corr = df[df['Class']==1].drop(['Time','Class'], axis=1).corr()
fig, ax = plt.subplots(figsize=(14, 10))
sns.heatmap(fraud_corr, cmap='RdBu_r', center=0, ax=ax,
            linewidths=0.3, square=True, cbar_kws={'shrink': 0.7})
ax.set_title('Feature Correlation — Fraud Transactions')
plt.tight_layout()
plt.show()

## 3. Feature Engineering

In [ ]:
df_eng = engineer_features(df)
print('New features added:')
new_cols = [c for c in df_eng.columns if c not in df.columns]
print(new_cols)

In [ ]:
# Visualise engineered features
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for ax, col, title in zip(axes,
    ['log_amount', 'pca_magnitude', 'hour_of_day'],
    ['Log Amount', 'PCA Magnitude', 'Hour of Day']):

    for cls, lbl, clr in [(0,'Legitimate','#2563EB'), (1,'Fraud','#DC2626')]:
        ax.hist(df_eng[df_eng['Class']==cls][col], bins=40,
                alpha=0.6, label=lbl, color=clr, density=True)
    ax.set_title(title)
    ax.legend()

plt.tight_layout()
plt.show()

## 4. Train / Test Split & Scaling

In [ ]:
feature_cols = [c for c in df_eng.columns if c not in ['Class', 'Time']]
X = df_eng[feature_cols]
y = df_eng['Class']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

scaler = StandardScaler()
scale_cols = ['Amount', 'log_amount', 'amount_zscore', 'pca_magnitude']
for col in scale_cols:
    X_train = X_train.copy()
    X_test = X_test.copy()
    X_train[col] = scaler.fit_transform(X_train[[col]])
    X_test[col] = scaler.transform(X_test[[col]])

print(f'Train size: {X_train.shape}, Test size: {X_test.shape}')
print(f'Fraud in train: {y_train.sum()}, Fraud in test: {y_test.sum()}')

## 5. Isolation Forest — Anomaly Scores

In [ ]:
iso = IsolationForest(n_estimators=200, contamination=0.002, random_state=42, n_jobs=-1)
iso.fit(X_train)

X_train = X_train.copy()
X_test = X_test.copy()
X_train['iso_score'] = iso.decision_function(X_train)
X_test['iso_score'] = iso.decision_function(X_test)
X_train['iso_flag'] = (iso.predict(X_train.drop('iso_score', axis=1)) == -1).astype(int)
X_test['iso_flag'] = (iso.predict(X_test.drop('iso_score', axis=1)) == -1).astype(int)

# How well does the anomaly score separate classes?
fig, ax = plt.subplots(figsize=(9, 4))
train_eval = X_train.copy()
train_eval['label'] = y_train.values

for cls, lbl, clr in [(0,'Legitimate','#2563EB'), (1,'Fraud','#DC2626')]:
    subset = train_eval[train_eval['label']==cls]['iso_score']
    ax.hist(subset, bins=60, alpha=0.6, label=lbl, color=clr, density=True)

ax.set_xlabel('Isolation Forest Score (lower = more anomalous)')
ax.set_ylabel('Density')
ax.set_title('Isolation Forest Score Distribution')
ax.legend()
plt.tight_layout()
plt.show()

## 6. SMOTE + XGBoost Training

In [ ]:
smote = SMOTE(sampling_strategy=0.1, random_state=42, k_neighbors=5)
X_res, y_res = smote.fit_resample(X_train, y_train)

print(f'Before SMOTE → Fraud: {y_train.sum()}, Legit: {(y_train==0).sum()}')
print(f'After SMOTE  → Fraud: {(y_res==1).sum()}, Legit: {(y_res==0).sum()}')

In [ ]:
scale_pos_weight = (y_res == 0).sum() / (y_res == 1).sum()

model = XGBClassifier(
    n_estimators=500,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    scale_pos_weight=scale_pos_weight,
    eval_metric='aucpr',
    random_state=42,
    use_label_encoder=False,
    n_jobs=-1,
    early_stopping_rounds=30
)

X_tr, X_val, y_tr, y_val = train_test_split(
    X_res, y_res, test_size=0.1, stratify=y_res, random_state=42
)

model.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], verbose=100)
print(f'Best iteration: {model.best_iteration}')

## 7. Threshold Optimisation & Evaluation

In [ ]:
probs = model.predict_proba(X_test)[:, 1]
precisions, recalls, thresholds = precision_recall_curve(y_test, probs)

# Find threshold maximising recall with precision >= 10%
best_t, best_r = 0.5, 0
for p, r, t in zip(precisions, recalls, thresholds):
    if p >= 0.10 and r > best_r:
        best_r, best_t = r, t

print(f'Optimal threshold: {best_t:.4f} | Recall: {best_r:.4f}')

preds = (probs >= best_t).astype(int)
print(classification_report(y_test, preds, target_names=['Legitimate', 'Fraud']))
print(f'ROC-AUC: {roc_auc_score(y_test, probs):.4f}')
print(f'Avg Precision (AUPR): {average_precision_score(y_test, probs):.4f}')

## 8. Final Visualisations

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Confusion Matrix
cm = confusion_matrix(y_test, preds)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Legit', 'Fraud'], yticklabels=['Legit', 'Fraud'], ax=axes[0])
axes[0].set_title('Confusion Matrix')
axes[0].set_xlabel('Predicted'); axes[0].set_ylabel('Actual')

# ROC Curve
fpr, tpr, _ = roc_curve(y_test, probs)
axes[1].plot(fpr, tpr, '#2563EB', lw=2, label=f'AUC={roc_auc_score(y_test, probs):.4f}')
axes[1].plot([0,1],[0,1],'k--'); axes[1].legend()
axes[1].set_title('ROC Curve'); axes[1].set_xlabel('FPR'); axes[1].set_ylabel('TPR')

# Feature Importance
fi = pd.Series(model.feature_importances_, index=X_train.columns).nlargest(15)
fi.sort_values().plot(kind='barh', ax=axes[2], color='#7C3AED')
axes[2].set_title('Top 15 Feature Importances')

plt.tight_layout()
plt.savefig('../outputs/final_summary.png', dpi=150, bbox_inches='tight')
plt.show()